# Qwen3.5-4B: 300 questions


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.22.1", "accelerate==1.15.0"],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==5.22.1', 'accelerate==1.15.0'], returncode=0)

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
from pathlib import Path
import shutil
import time
import traceback
import zipfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

import torch
from transformers import (
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    Qwen3_5ForConditionalGeneration,
    set_seed,
)

run_name = os.environ.get("PLANT_RUN_NAME", "qwen35_4b_nonthinking")
dataset_path = Path("/content/data/prepared/plant_care_pilot/questions.jsonl")
run_dir = Path("/content/data/runs/plant_care_pilot") / run_name
config = {
    "model_id": "Qwen/Qwen3.5-4B",
    "model_revision": "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    "dataset_name": "plant_care_pilot",
    "enable_thinking": False,
    "dtype": "float16",
    "attention_implementation": "sdpa",
    "seed": 42,
    "seed_policy": "seed + one-based position in questions.jsonl",
    "max_input_tokens": 1024,
    "generation": {
        "max_new_tokens": 512,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.8,
        "top_k": 20,
        "repetition_penalty": 1.0,
    },
    "presence_penalty": 1.5,
    "system_prompt": (
        "Ты помощник по уходу за комнатными растениями, садом и огородом. "
        "Отвечай новичку на русском языке кратко и понятно, объясняй причины советов. "
        "Если совет зависит от недостающих условий, задавай уточняющие вопросы. "
        "Обозначай неопределённость и не выдумывай сведения."
    ),
}
print(run_name)

qwen35_4b_nonthinking


In [ ]:
if not dataset_path.exists():
    raise FileNotFoundError("Upload questions.jsonl before running this cell.")

dataset_bytes = dataset_path.read_bytes()
questions = [json.loads(line) for line in dataset_bytes.decode().splitlines() if line.strip()]
if len(questions) != 300 or len({row["id"] for row in questions}) != 300:
    raise ValueError("Expected 300 unique questions.")
seed_by_id = {row["id"]: config["seed"] + i for i, row in enumerate(questions, 1)}
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0):
    raise RuntimeError("This notebook requires an NVIDIA T4.")

run_dir.mkdir(parents=True, exist_ok=True)
config_path = run_dir / "config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise ValueError("This run already has different settings. Choose a new run name.")
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n")
shutil.copy2(dataset_path, run_dir / "questions.jsonl")

environment = {
    "started_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gib": torch.cuda.get_device_properties(0).total_memory / 2**30,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "cuda": torch.version.cuda,
    "packages": {name: version(name) for name in [
        "torch", "transformers", "accelerate", "tokenizers", "safetensors", "huggingface-hub"
    ]},
    "nvidia_smi": subprocess.check_output(["nvidia-smi"], text=True),
}
(run_dir / "environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2) + "\n"
)
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
print(json.dumps(environment, ensure_ascii=False, indent=2))

{
  "started_at_utc": "2026-11-02T07:10:49.114784+00:00",
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "gpu": "Tesla T4",
  "gpu_memory_gib": 14.56317138671875,
  "compute_capability": [
    7,
    5
  ],
  "cuda": "13.0",
  "packages": {
    "torch": "2.11.0+cu130",
    "transformers": "5.22.1",
    "accelerate": "1.15.0",
    "tokenizers": "0.23.3",
    "safetensors": "0.8.0",
    "huggingface-hub": "1.33.0"
  },
  "nvidia_smi": "Mon Nov  2 07:10:49 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |\n+-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |\n| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |\n|                                         |

In [ ]:
load_started = time.perf_counter()
torch.cuda.reset_peak_memory_stats()
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"], revision=config["model_revision"], token=False
)
model = Qwen3_5ForConditionalGeneration.from_pretrained(
    config["model_id"],
    revision=config["model_revision"],
    dtype=torch.float16,
    device_map={"": 0},
    attn_implementation=config["attention_implementation"],
    use_safetensors=True,
    token=False,
)
model.eval()
load_stats = {
    "load_seconds": time.perf_counter() - load_started,
    "allocated_gib": torch.cuda.memory_allocated() / 2**30,
    "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    "model_class": type(model).__name__,
}
(run_dir / "model-loading.json").write_text(json.dumps(load_stats, indent=2) + "\n")
print(json.dumps(load_stats, indent=2))

{
  "load_seconds": 110.42129349300001,
  "allocated_gib": 8.455068588256836,
  "peak_reserved_gib": 8.46484375,
  "model_class": "Qwen3_5ForConditionalGeneration"
}


In [ ]:
class PresencePenalty(LogitsProcessor):
    def __init__(self, penalty, prompt_length):
        self.penalty = penalty
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores):
        generated = input_ids[:, self.prompt_length:]
        if generated.numel():
            seen = torch.zeros_like(scores, dtype=torch.bool)
            seen.scatter_(1, generated, True)
            scores = scores - seen.to(scores.dtype) * self.penalty
        return scores


answers_path = run_dir / "answers.jsonl"
answers = [json.loads(line) for line in answers_path.read_text().splitlines()] if answers_path.exists() else []
if len({row["id"] for row in answers}) != len(answers):
    raise ValueError("Duplicate question IDs in saved answers.")


def save_artifacts():
    ordered = sorted(answers, key=lambda row: row["id"])
    summary = {
        "run_name": run_name,
        "completed": len(ordered),
        "expected": len(questions),
        "output_tokens": sum(row["output_tokens"] for row in ordered),
        "generation_seconds": sum(row["elapsed_seconds"] for row in ordered),
        "truncated_ids": [row["id"] for row in ordered if row["finish_reason"] == "length"],
        "thinking_marker_ids": [row["id"] for row in ordered if row["thinking_markers"]],
        "peak_reserved_gib": max([load_stats["peak_reserved_gib"]] + [row["peak_reserved_gib"] for row in ordered]),
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    sections = ["# Qwen3.5-4B answers", f"Run: `{run_name}`. Non-thinking mode."]
    for row in ordered:
        sections.extend([f"## Question {row['id']}", row["question"], row["answer"]])
    (run_dir / "answers.md").write_text("\n\n".join(sections) + "\n")
    with zipfile.ZipFile(run_dir / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_dir.iterdir()):
            if path.is_file() and path.name != "artifacts.zip":
                archive.write(path, arcname=path.name)


def answer_question(row):
    if row["id"] in {answer["id"] for answer in answers}:
        return
    seed = seed_by_id[row["id"]]
    set_seed(seed)
    messages = [
        {"role": "system", "content": config["system_prompt"]},
        {"role": "user", "content": row["question"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to("cuda:0")
    input_length = inputs["input_ids"].shape[1]
    if input_length > config["max_input_tokens"]:
        raise ValueError(f"Question {row['id']} exceeds the input token limit.")
    penalty = PresencePenalty(config["presence_penalty"], input_length)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **config["generation"],
            logits_processor=LogitsProcessorList([penalty]),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = output[0, input_length:]
    raw_response = tokenizer.decode(generated, skip_special_tokens=False)
    eos_ids = model.generation_config.eos_token_id
    eos_ids = eos_ids if isinstance(eos_ids, list) else [eos_ids]
    finished = bool(generated.numel() and generated[-1].item() in eos_ids)
    answer = {
        "id": row["id"],
        "question": row["question"],
        "messages": messages,
        "rendered_prompt": prompt,
        "answer": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "raw_response": raw_response,
        "generation_seed": seed,
        "input_tokens": input_length,
        "output_tokens": generated.numel(),
        "elapsed_seconds": elapsed,
        "tokens_per_second": generated.numel() / elapsed,
        "finish_reason": "eos" if finished else "length",
        "thinking_markers": "<think>" in raw_response or "</think>" in raw_response,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 2**30,
    }
    with answers_path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(answer, ensure_ascii=False) + "\n")
    answers.append(answer)
    save_artifacts()
    print(f"{len(answers)}/{len(questions)} {row['id']}: {answer['output_tokens']} tokens, {elapsed:.1f} s, {answer['finish_reason']}", flush=True)
    print(answer["answer"], flush=True)

In [ ]:
try:
    answer_question(questions[0])
except Exception:
    (run_dir / "error.txt").write_text(traceback.format_exc())
    save_artifacts()
    raise

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


1/300 pcp-garden-001: 323 tokens, 24.5 s, eos


Нет, поливать растения после захода солнца или в сумерках **не вредно**. Напротив, это считается лучшим временем для полива.

**Причины:**
1.  **Предотвращение гниения корней:** Днем на листьях и земле может быть высокая температура и влажность от прямого солнца. Если полить растение в этот период, вода быстро испарится, но оставшаяся влага под солнечными лучами может сжечь нежные корни или вызвать грибковые заболевания (например, корневую гниль). В сумерках воздух прохладнее, а риск перегрева минимален.
2.  **Снижение риска ожогов:** Влажные листья под ярким солнцем нагреваются быстрее сухих и могут получить химический ожог от солнечных лучей. Полив вечером позволяет листьям высохнуть до наступления следующего дня без воздействия света.
3.  **Эффективность усвоения:** Ночью и утром корни активнее «просыпаются» и лучше поглощают воду и питательные вещества, так как транспирация (процесс испарения воды через листья) снижается.

**Важное уточнение:**
Если вы живете в регионе с очень жарк

In [ ]:
try:
    for row in questions[1:]:
        answer_question(row)
except Exception:
    (run_dir / "error.txt").write_text(traceback.format_exc())
    raise
finally:
    save_artifacts()

print((run_dir / "summary.json").read_text())
print(f"Results archive: {run_dir / 'artifacts.zip'}")

2/300 pcp-garden-002: 512 tokens, 33.5 s, length


Применять строгий севооборот в условиях неравномерного освещения и тепла — это отличный подход, так как он позволяет адаптировать агротехнику под ваши реальные условия.

**Ключевой принцип:** Не дробите грядки на равные квадраты. Делайте «зоны» по факторам среды (свет/тепло), а внутри зон соблюдайте правило **смены культур**.

### 1. Где посадить ранние салаты?
Ранние салаты относятся к семейству **капуста**, но они очень требовательны к свету и температуре (особенно при высадке рассадой).
*   **Рекомендация:** Посаживать их нужно именно на то самое **самое теплое и освещенное место**, куда вы хотите отправить морковь и редис.
*   **Почему:** Холодный или тенистый салат быстро вытягивается, становится волокнистым и может не успеть созреть. Теплый воздух ускоряет рост листьев.

### 2. Как применять севооборот на практике?
Разбейте огород не на участки «под корнеплоды», а на зоны по типу использования:

*   **Зона А (Теплое + Светлое):**
    *   *Что растет:* Ранние салаты, редис, морков

3/300 pcp-garden-003: 265 tokens, 17.5 s, eos


Да, это **стрелкование** (создание цветочного стебля). Салаты — холодостойкие растения, и если им не хватает тепла или света в текущий момент, они могут попытаться размножаться семенами, даже если температура воздуха комфортная.

**Можно ли есть?**
*   **Стебель:** Нет, его лучше удалить сразу, чтобы растение не тратило силы на цветение и не становилось жестким.
*   **Листья:** Да, листья обычно остаются съедобными и мягкими, так как стрелкование часто происходит из-за внешних условий, а не отравления.

**Важное уточнение:**
Если вы видите желтые пятна на листьях рядом с цветоносом, это может быть признак болезни (например, фузариоза), тогда лист нужно будет убрать целиком. Если же листья зеленые и упругие — смело используйте их.

Чтобы предотвратить повторное стрелкование, проверьте:
1.  Достаточно ли свету (солнечных часов)?
2.  Не слишком ли холодно?
3.  Правильно ли поливаете (переувлажнение часто вызывает стрелкование)?


4/300 pcp-garden-004: 265 tokens, 17.3 s, eos


Для ревеня лучше всего **выдергивать** черешки руками или аккуратно срезать их, но **не нужно оставлять часть стебля**.

**Почему так:**
*   **Выдергивание:** Это самый простой способ. Ревень — это корневищное растение, и его «стебель» (корневище) находится глубоко под землей. Если вы просто выдернете лист у основания, корневая система не пострадает.
*   **Оставлять часть:** Не рекомендуется. Если оставить верхушку листа, она будет продолжать расти и забирать питательные вещества из корней, что замедлит рост новых листьев и может привести к ослаблению куста.
*   **Срезание:** Можно использовать ножницы, но делайте срез строго у самого основания (где лист встречается с почкой), чтобы не повредить точку роста.

**Совет по сбору:**
Чтобы куст рос гуще и давал больше урожая в будущем, старайтесь собирать урожай регулярно, не допуская перерастания листьев. В идеале оставляйте на растении только 3–4 самых нижних крупных листа для защиты почек от мороза зимой, а остальные убирайте.


... (589 outputs left out)

Скорее всего, это не наследственная особенность конкретного растения, а признак того, что ему не хватает ресурсов для формирования полноценного цветочного бутона.

**Причины и объяснение:**
1.  **Недостаток фосфора.** Фосфор отвечает за цветение и образование бутонов. Без него растение тратит все силы на выживание и быстро сбрасывает цветы.
2.  **Стресс или дефицит света.** Если свет слишком слабый или есть резкие перепады температуры/влажности, цветок может цвести короткими «спорадическими» ветками, чтобы сохранить энергию.
3.  **Перегрузка удобрениями.** Иногда избыток азота (для роста листьев) мешает формированию цветов.

**Что делать?**
Попробуйте добавить в воду для полива удобрение с высоким содержанием фосфора (P), например, «Фертика Лайт» или специализированное удобрение для орхидей в фазе цветения (обычно соотношение N-P-K: 0-5-6). Также проверьте, стоит ли растение на самом светлом месте, куда проникает утренний свет (не прямые полуденные лучи).

*Если после подкормки через м

300/300 pcp-indoor-150: 252 tokens, 16.1 s, eos


Скорее всего, орхидея не зацветет в этих условиях.

**Причина:** Орхидеи (особенно фаленопсис) требуют **сильный перепад температур** между днем и ночью (минимум 5–7°C) и длительную фазу покоя для формирования цветочных почек. Холодильник дает низкую температуру, но не создает нужного цикла дня и ночи, а отсутствие дневного света блокирует фотосинтез, необходимый для роста новых цветоносов. Кроме того, постоянный холод может вызвать гниение корней.

**Что делать:**
1. Переместите растение в светлое место с хорошим доступом к естественному свету (окно).
2. Убедитесь, что рядом нет сквозняков и прямых лучей солнца летом.
3. Если нужно искусственное освещение, используйте фитолампу или обычную лампу накаливания/LED, включенную на 6–8 часов в день.

*Примечание: Без смены места на светлое и создания температурного контраста цветение гарантированно не наступит.*


{
  "run_name": "qwen35_4b_nonthinking",
  "completed": 300,
  "expected": 300,
  "output_tokens": 105012,
  "generation_seconds": 6768.0250460180105,
  "truncated_ids": [
    "pcp-garden-002",
    "pcp-garden-012",
    "pcp-garden-076",
    "pcp-garden-101",
    "pcp-garden-108",
    "pcp-garden-113",
    "pcp-garden-131",
    "pcp-garden-149",
    "pcp-indoor-006",
    "pcp-indoor-012",
    "pcp-indoor-015",
    "pcp-indoor-020",
    "pcp-indoor-022",
    "pcp-indoor-035",
    "pcp-indoor-040",
    "pcp-indoor-047",
    "pcp-indoor-049",
    "pcp-indoor-060",
    "pcp-indoor-070",
    "pcp-indoor-077",
    "pcp-indoor-085",
    "pcp-indoor-087",
    "pcp-indoor-096",
    "pcp-indoor-103",
    "pcp-indoor-105",
    "pcp-indoor-115",
    "pcp-indoor-147",
    "pcp-indoor-148"
  ],
  "thinking_marker_ids": [],
  "peak_reserved_gib": 8.646484375
}

Results archive: /content/data/runs/plant_care_pilot/qwen35_4b_nonthinking/artifacts.zip
